In [12]:
from ollama import Client

client = Client(host="http://localhost:11434")
# 11434 port utilisé par defualt par Ollama

# Lancer le server python
# python -m uvicorn main:app --host 0.0.0.0 --port 8000

In [13]:
response = client.chat(
    model="qwen2.5:1.5B",
    messages=[
        {"role": "user", "content": "Bonjour ! Présente-toi en une phrase."}
    ],
)

In [14]:
print(response.message.content)

Bonjour, je suis Qwen, une assistant virtuel conçu par Alibaba Cloud pour aider les utilisateurs à obtenir les informations dont ils ont besoin.


In [15]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from ollama import AsyncClient, ResponseError
from pydantic import BaseModel, Field

### SETUP API

In [16]:
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000",
        "http://localhost:5173",
    ],
    allow_methods=["POST"],
    allow_headers=["Content-Type"],
)

client = AsyncClient(
    host="http://localhost:11434",
    timeout=120.0,
)


In [17]:
client = AsyncClient(
    host="http://localhost:11434",
    timeout=120.0,
)

In [18]:
MODEL = "qwen2.5:1.5B"

In [19]:
class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=10000)


In [ ]:
@app.post("/chat")
async def chat(body: ChatRequest):
    try:
        response = await client.chat(
            model=MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "If your name is asqued, answer Axel, never Qwen",
                },
                {"role": "user", "content": body.message},
            ],
            stream=False,
        )

        return {"reply": response.message.content}

    except ResponseError as exc:
        raise HTTPException(
            status_code=502,
            detail="Ollama not able to generate response",
        ) from exc

    except ConnectionError as exc:
        raise HTTPException(
            status_code=503,
            detail="Ollama no accessible",
        ) from exc